# 02 - Feature Engineering & Risk Labeling

This notebook does two things: engineers modeling features
(`Customer_Age`, `Vehicle_Age`, `Age_Group`, `Income_Quartile`,
`Has_Claim`) from the raw columns, and builds the `Risk_Category` proxy
label used to train the risk classifier in notebook 03.

Why a proxy label at all? The raw export has no ground-truth underwriting
decision column. Rather than pretend otherwise, this builds a
transparent, documented scoring rule from known risk factors (see
`src/risk_labeling.py` and `config.RISK_WEIGHTS`), calibrated so the
resulting three-way split looks like a realistic insurance book of
business. That's a real limitation worth being upfront about rather than
glossing over.


In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
from src.data_prep import load_raw_data, clean_data
from src.features import engineer_features, compute_risk_components
from src.risk_labeling import compute_raw_risk_score, assign_risk_category
from src import config

df = engineer_features(clean_data(load_raw_data()))
df[['BirthDate', 'Customer_Age', 'Car_Year', 'Vehicle_Age', 'Age_Group', 'Income_Quartile', 'Has_Claim']].head()


[data_prep] Dropped 1 duplicate policy rows.


,BirthDate,Customer_Age,Car_Year,Vehicle_Age,Age_Group,Income_Quartile,Has_Claim
0,1965-08-09,58.4,2017,7,56-65,4,1
1,1991-04-21,32.7,2011,13,26-35,1,1
2,2002-03-08,21.8,2000,24,18-25,1,0
3,1962-05-10,61.6,2007,17,56-65,3,1
4,1995-01-15,29.0,2013,11,26-35,2,1


## Risk components: one normalized 0-1 score per weighted factor

In [3]:
components = compute_risk_components(df)
components.describe().T


,count,mean,std,min,25%,50%,75%,max
age_risk,37530.0,0.281789,0.263848,0.00,0.050740,0.206048,0.456662,1.00
kids_driving_risk,37530.0,0.139444,0.245676,0.00,0.000000,0.000000,0.333333,1.00
income_risk,37530.0,0.501172,0.288552,0.00,0.252184,0.503897,0.750411,1.00
car_year_risk,37530.0,0.195107,0.137348,0.00,0.092308,0.169231,0.276923,1.00
vehicle_age_risk,37530.0,0.195107,0.137348,0.00,0.092308,0.169231,0.276923,1.00
car_use_risk,37530.0,0.199334,0.399505,0.00,0.000000,0.000000,0.000000,1.00
parent_risk,37530.0,0.442446,0.496683,0.00,0.000000,0.000000,1.000000,1.00
coverage_zone_risk,37530.0,0.570926,0.273045,0.25,0.350000,0.500000,0.750000,1.00
education_risk,37530.0,0.656379,0.283784,0.00,0.660000,0.660000,1.000000,1.00
gender_risk,37530.0,0.499547,0.250003,0.25,0.250000,0.250000,0.750000,0.75


## Weights, ordered by how much each factor plausibly drives risk

In [4]:
import pandas as pd
pd.Series(config.RISK_WEIGHTS, name='weight').sort_values(ascending=False)


age_risk              0.262
kids_driving_risk     0.214
income_risk           0.128
car_year_risk         0.105
vehicle_age_risk      0.098
car_use_risk          0.076
parent_risk           0.052
coverage_zone_risk    0.031
education_risk        0.024
gender_risk           0.010
Name: weight, dtype: float64

In [5]:
df['risk_score_raw'] = compute_raw_risk_score(df)
df[config.RISK_TARGET] = assign_risk_category(df)

df[config.RISK_TARGET].value_counts(normalize=True).round(3)


Risk_Category
Low       0.636
Medium    0.248
High      0.116
Name: proportion, dtype: float64

That split happens by construction, since the labels are bucketed on
score quantiles. What actually matters is whether the raw features, not
the score itself, can recover this label reasonably well. That's tested
in notebook 03.


In [6]:
df.groupby(config.RISK_TARGET)['risk_score_raw'].describe()


,count,mean,std,min,25%,50%,75%,max
Risk_Category,,,,,,,,
High,4354.0,0.475562,0.079874,0.218046,0.420515,0.469485,0.525637,0.842035
Low,23869.0,0.221933,0.073349,0.023900,0.168480,0.220163,0.273523,0.503200
Medium,9307.0,0.353684,0.064685,0.155261,0.310674,0.353213,0.397022,0.585657
